# IBKR Option Chain Fetcher

Fetches live option chains from Interactive Brokers (IB Gateway / TWS) via `ib_insync`
and saves them to the local SQLite database used by the rest of the skew framework.

## Prerequisites
- IB Gateway (paper or live) running on `127.0.0.1:4002` (paper) or `127.0.0.1:4001` (live)
- Required packages:
  ```
  pip install ib_insync nest_asyncio
  ```
- In IB Gateway: API → Settings → Enable ActiveX and Socket Clients ✓

## Sections
1. **Connect** – establish IB Gateway connection
2. **Live snapshot** – fetch current chain for one ticker and save to DB
3. **Historical backfill** – pull N trading days of IV history per contract
4. **Disconnect**

In [ ]:
# ── 0. Imports ──────────────────────────────────────────────────────────────
import sys, time, math
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# nest_asyncio lets ib_insync's sync wrappers work inside Jupyter's event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    raise ImportError("Install nest_asyncio:  pip install nest_asyncio")

# Project root on path
ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from skew.data_store import save_option_snapshot, list_snapshots

try:
    from ib_insync import IB, Stock, Option, util
    print("ib_insync OK")
except ImportError:
    raise ImportError("Install ib_insync:  pip install ib_insync")

---
## 1. Configuration

In [ ]:
# ── 1. Config ───────────────────────────────────────────────────────────────
IB_HOST   = "127.0.0.1"
IB_PORT   = 4002          # 4002 = paper gateway, 4001 = live gateway
CLIENT_ID = 10            # must be unique per active IB connection

DB_PATH   = "../data/options.db"

# Tickers to fetch (exchange + currency inferred automatically)
TICKERS   = ["NVDA", "MSFT"]

# Liquidity filters applied to each chain
MIN_BID        = 0.05    # discard contracts with bid < this
MAX_SPREAD_PCT = 0.25    # discard if (ask-bid)/mid > this
MIN_OPEN_INT   = 10      # discard contracts with open interest < this

# Expiry range
MIN_EXPIRY_DAYS = 7      # ignore very near-term contracts
MAX_EXPIRY_DAYS = 365    # ignore options expiring beyond 1 year

# Strike selection per expiry
N_STRIKES_PER_SIDE = 10  # OTM call strikes above ATM + OTM put strikes below ATM
                          # total per expiry = 2*N_STRIKES_PER_SIDE + 1 (ATM)

# How many calendar days back to backfill (Section 4)
BACKFILL_DAYS  = 30

---
## 2. Connect to IB Gateway

In [ ]:
# ── 2. Connect ──────────────────────────────────────────────────────────────
ib = IB()
ib.connect(IB_HOST, IB_PORT, clientId=CLIENT_ID, readonly=True)
print(f"Connected: {ib.isConnected()}  |  account: {ib.managedAccounts()}")

---
## 3. Live Option Chain Snapshot

In [ ]:
# ── Helper: fetch chain for one ticker ──────────────────────────────────────
def _get_spot(ib: IB, ticker: str) -> float:
    """Return last trade price for the underlying equity."""
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    [td] = ib.reqTickers(stk)
    spot = td.last if td.last and td.last > 0 else td.close
    if not spot or spot <= 0:
        raise ValueError(f"Cannot get spot for {ticker}")
    return float(spot)


def _select_strikes(all_strikes: list, spot: float, n_per_side: int) -> list:
    """
    From a sorted list of available strikes, return:
      - the ATM strike (closest to spot)
      - n_per_side strikes above ATM, equally spaced across the OTM call range
      - n_per_side strikes below ATM, equally spaced across the OTM put range
    """
    s = sorted(all_strikes)
    atm_idx = min(range(len(s)), key=lambda i: abs(s[i] - spot))

    above = s[atm_idx + 1:]
    below = s[:atm_idx]

    def _pick(lst, n):
        if len(lst) <= n:
            return lst
        idx = np.round(np.linspace(0, len(lst) - 1, n)).astype(int)
        return [lst[i] for i in idx]

    chosen = sorted(set(_pick(below, n_per_side) + [s[atm_idx]] + _pick(above, n_per_side)))
    return chosen


def _build_chain_df(
    ib: IB,
    ticker: str,
    spot: float,
    min_bid: float        = MIN_BID,
    max_spread_pct: float = MAX_SPREAD_PCT,
    min_oi: int           = MIN_OPEN_INT,
    n_per_side: int       = N_STRIKES_PER_SIDE,
    min_expiry_days: int  = MIN_EXPIRY_DAYS,
    max_expiry_days: int  = MAX_EXPIRY_DAYS,
) -> pd.DataFrame:
    """
    Fetch a live option chain snapshot from IBKR for *ticker*.

    Per expiry fetches ATM + n_per_side equally-spaced OTM calls
    + n_per_side equally-spaced OTM puts.

    Returns a normalised DataFrame ready for save_option_snapshot.
    """
    # 1. Get contract parameters
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    chains = ib.reqSecDefOptParams(ticker, "", "STK", stk.conId)
    if not chains:
        raise RuntimeError(f"No option chain found for {ticker}")

    chain = max(chains, key=lambda c: len(c.strikes))

    # 2. Filter expiries to [min_expiry_days, max_expiry_days]
    today = pd.Timestamp.today().normalize()
    expiries = sorted(
        e for e in chain.expirations
        if min_expiry_days < (pd.to_datetime(e) - today).days <= max_expiry_days
    )

    # 3. Build contract objects: ATM + n_per_side per side, per expiry
    all_strikes = sorted(chain.strikes)
    contracts = []
    for exp in expiries:
        selected = _select_strikes(all_strikes, spot, n_per_side)
        for k in selected:
            for right in ("C", "P"):
                contracts.append(Option(ticker, exp, k, right, "SMART"))

    print(f"  {ticker}: {len(expiries)} expiries × {2*n_per_side+1} strikes × 2 = {len(contracts)} contracts")

    # Qualify in batches of 200
    qualified = []
    for i in range(0, len(contracts), 200):
        qualified.extend(ib.qualifyContracts(*contracts[i:i+200]))
        time.sleep(0.2)

    if not qualified:
        raise RuntimeError(f"No qualified contracts for {ticker}")

    # 4. Request market data snapshots (includes modelGreeks)
    CHUNK = 100
    tickers_data = []
    for i in range(0, len(qualified), CHUNK):
        batch = qualified[i:i+CHUNK]
        tds   = ib.reqTickers(*batch)
        tickers_data.extend(tds)
        time.sleep(1.0)

    # 5. Parse into rows
    rows = []
    for td in tickers_data:
        c   = td.contract
        bid = td.bid  if td.bid  and td.bid  > 0 else np.nan
        ask = td.ask  if td.ask  and td.ask  > 0 else np.nan
        mid = (bid + ask) / 2 if (not np.isnan(bid) and not np.isnan(ask)) else np.nan

        iv    = np.nan
        delta = np.nan
        gamma = np.nan
        vega  = np.nan
        theta = np.nan
        if td.modelGreeks:
            g = td.modelGreeks
            iv    = g.impliedVol  if g.impliedVol  and g.impliedVol > 0  else np.nan
            delta = g.delta       if g.delta  is not None else np.nan
            gamma = g.gamma       if g.gamma  is not None else np.nan
            vega  = g.vega        if g.vega   is not None else np.nan
            theta = g.theta       if g.theta  is not None else np.nan

        oi = td.callOpenInterest if c.right == "C" else td.putOpenInterest
        if oi is None:
            oi = np.nan

        rows.append({
            "strike":           float(c.strike),
            "expiry":           c.lastTradeDateOrContractMonth,
            "option_type":      "call" if c.right == "C" else "put",
            "bid":              bid,
            "ask":              ask,
            "option_price":     mid,
            "implied_vol":      iv,
            "delta":            delta,
            "gamma":            gamma,
            "vega":             vega,
            "theta":            theta,
            "open_interest":    oi,
            "underlying_price": spot,
        })

    df = pd.DataFrame(rows)

    # 6. Liquidity filters
    df = df.dropna(subset=["bid", "ask", "implied_vol"])
    df = df[df["bid"] >= min_bid]
    spread_pct = (df["ask"] - df["bid"]) / df["option_price"].clip(lower=1e-6)
    df = df[spread_pct <= max_spread_pct]
    if min_oi > 0:
        df = df[df["open_interest"].fillna(0) >= min_oi]

    # Normalise expiry to YYYY-MM-DD
    df["expiry"] = pd.to_datetime(df["expiry"], format="%Y%m%d").dt.date.astype(str)

    return df.reset_index(drop=True)

In [ ]:
# ── 3. Fetch live chain and save ─────────────────────────────────────────────
today_str = date.today().isoformat()

for tkr in TICKERS:
    print(f"\n{'='*60}")
    print(f"Fetching {tkr} …")
    try:
        spot  = _get_spot(ib, tkr)
        print(f"  spot = {spot:.2f}")
        chain = _build_chain_df(ib, tkr, spot)
        print(f"  raw rows after filters: {len(chain)}")

        n = save_option_snapshot(chain, tkr, db_path=DB_PATH, source="ibkr")
        print(f"  saved {n} new rows  (snapshot_date={today_str})")

    except Exception as e:
        print(f"  ERROR: {e}")

print("\nDone. DB summary:")
display(list_snapshots(db_path=DB_PATH))

---
## 4. Historical IV Backfill

IBKR `reqHistoricalData` with `whatToShow='OPTION_IMPLIED_VOLATILITY'` returns a
daily bar series (open/high/low/close) of the IV for a specific option contract.

Strategy here:
1. Get today's live chain to know which contracts exist
2. For each contract, request N days of IV history
3. For each historical date, construct a synthetic row and call `save_option_snapshot`
   with `snapshot_date=` override

> **Rate limits**: IBKR allows 50 historical data requests per 10 seconds.
> We enforce a `HIST_DELAY` between requests. For large chains (1000+ contracts)
> this can take several minutes.

In [ ]:
# ── 4a. Config for backfill ──────────────────────────────────────────────────
BACKFILL_TICKER = "NVDA"   # ticker to backfill (one at a time to control rate)
BACKFILL_DAYS   = 30       # calendar days back from today
HIST_DELAY      = 0.22     # seconds between historical requests (≤50/10 s limit)

# Further narrow strikes for backfill to reduce request count
BACKFILL_SPOT_RANGE = 0.35  # ±35% moneyness band

In [ ]:
# ── 4b. Build list of contracts to backfill ──────────────────────────────────
print(f"Getting contract list for {BACKFILL_TICKER} …")
bf_spot  = _get_spot(ib, BACKFILL_TICKER)
bf_chain = _build_chain_df(
    ib, BACKFILL_TICKER, bf_spot,
    spot_range_pct=BACKFILL_SPOT_RANGE,
    min_bid=0.0, max_spread_pct=1.0, min_oi=0,   # no filters — just need contract list
)
print(f"  {len(bf_chain)} contracts to backfill")
bf_chain.head(3)

In [ ]:
# ── 4c. Fetch historical IV per contract ─────────────────────────────────────
# Builds a dict:  date_str → list of row dicts

stk_contract = Stock(BACKFILL_TICKER, "SMART", "USD")
ib.qualifyContracts(stk_contract)

duration_str = f"{BACKFILL_DAYS} D"
hist_rows: dict[str, list] = {}  # keyed by date string

errors = 0
for i, row in bf_chain.iterrows():
    exp_ibkr = row["expiry"].replace("-", "")  # YYYYMMDD
    right    = "C" if row["option_type"] == "call" else "P"
    opt_con  = Option(BACKFILL_TICKER, exp_ibkr, row["strike"], right, "SMART")
    try:
        ib.qualifyContracts(opt_con)
    except Exception:
        errors += 1
        continue

    try:
        bars = ib.reqHistoricalData(
            opt_con,
            endDateTime="",
            durationStr=duration_str,
            barSizeSetting="1 day",
            whatToShow="OPTION_IMPLIED_VOLATILITY",
            useRTH=True,
            formatDate=1,
        )
    except Exception as e:
        errors += 1
        time.sleep(HIST_DELAY)
        continue

    for bar in bars:
        d_str = str(bar.date)[:10]  # YYYY-MM-DD
        iv_close = bar.close        # daily closing IV
        if iv_close <= 0:
            continue
        hist_rows.setdefault(d_str, []).append({
            "strike":           row["strike"],
            "expiry":           row["expiry"],
            "option_type":      row["option_type"],
            "implied_vol":      iv_close,
            "underlying_price": bf_spot,   # today's spot as proxy
            # bid/ask/option_price unavailable from IV bars
            "bid":              np.nan,
            "ask":              np.nan,
            "option_price":     np.nan,
        })

    time.sleep(HIST_DELAY)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(bf_chain)} done  ({errors} errors)")

print(f"\nFinished: {len(hist_rows)} dates collected, {errors} errors")
print("Dates:", sorted(hist_rows.keys()))

In [ ]:
# ── 4d. Save historical rows to DB ───────────────────────────────────────────
total_saved = 0
for d_str, rows in sorted(hist_rows.items()):
    df_day = pd.DataFrame(rows)
    n = save_option_snapshot(
        df_day,
        BACKFILL_TICKER,
        db_path=DB_PATH,
        source="ibkr_hist",
        snapshot_date=d_str,   # override date for historical rows
    )
    total_saved += n
    print(f"  {d_str}: {len(df_day)} rows → {n} new")

print(f"\nTotal new rows saved: {total_saved}")
print("DB summary:")
display(list_snapshots(db_path=DB_PATH))

---
## 5. Disconnect

In [ ]:
ib.disconnect()
print("Disconnected.")

---
## Appendix: Column mapping reference

| IBKR field | DB column |
|---|---|
| `modelGreeks.impliedVol` | `implied_vol` |
| `modelGreeks.delta` | `delta` |
| `modelGreeks.gamma` | `gamma` |
| `modelGreeks.vega` | `vega` |
| `modelGreeks.theta` | `theta` |
| `bid` | `bid` |
| `ask` | `ask` |
| `(bid+ask)/2` | `option_price` |
| `contract.strike` | `strike` |
| `contract.lastTradeDateOrContractMonth` → YYYY-MM-DD | `expiry` |
| `"C"→"call", "P"→"put"` | `option_type` |
| `callOpenInterest` / `putOpenInterest` | `open_interest` |

Historical IV bars (`whatToShow='OPTION_IMPLIED_VOLATILITY'`) return only the IV series.
Bid/ask/price are stored as NaN for historical rows — they are not needed for skew analytics
(`pdf_calc.ipynb` uses `implied_vol` directly).